# Giai đoạn 3 — Baseline và kiểm tra thiếu modality (Đã tối ưu hóa tốc độ)

Notebook này thực hiện huấn luyện và đánh giá các mô hình Baseline (cả Sales-Only và Multimodal) trên tập dữ liệu chuẩn hóa.
Đồng thời kiểm tra độ bền vững trong các kịch bản thiếu hụt phương thức (Missing Modalities).

In [ ]:
from pathlib import Path
import os
import sys
import torch

# 1. Tự động xác định thư mục mã nguồn gốc (hỗ trợ cả Local và Kaggle)
CODE_ROOT_OVERRIDE = None
if CODE_ROOT_OVERRIDE:
    CODE_ROOT = Path(CODE_ROOT_OVERRIDE).resolve()
else:
    candidates = {p.resolve() for p in [Path.cwd(), Path.cwd().parent] if (p/'src/config.py').is_file()}
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            dirs[:] = [d for d in dirs if d not in {'images', '.git', '__pycache__'}]
            candidate = Path(root)
            if candidate.name == 'src' and 'config.py' in files and (candidate/'pipeline/train_baselines.py').is_file():
                candidates.add(candidate.parent.resolve())
    if len(candidates) == 0:
        CODE_ROOT = Path.cwd().resolve()
    else:
        CODE_ROOT = next(iter(candidates))

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.utils.paths import resolve_data_dir, resolve_raw_dir, resolve_embeddings_dir
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else CODE_ROOT
print('✅ Code Root:', CODE_ROOT)
print('✅ Output Dir:', WORK_ROOT)
print('🚀 Device:   ', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 2. Cấu hình Tham số Tối ưu hóa Tốc độ Huấn luyện
DATA_DIR_OVERRIDE = None
EMBEDDINGS_DIR_OVERRIDE = None

DATA_DIR = resolve_data_dir(CODE_ROOT, DATA_DIR_OVERRIDE)
EMBEDDINGS_DIR = resolve_embeddings_dir(CODE_ROOT, EMBEDDINGS_DIR_OVERRIDE)

# Danh mục mô hình đánh giá
MODELS = ['zero', 'last_value', 'moving_average', 'linear', 'mlp', 'lstm', 'transformer', 'multimodal_mlp']

# Tinh chỉnh tối ưu hóa thời gian (Chạy nhanh hơn 3-5 lần mà vẫn đủ đa dạng):
SEEDS = [42]                 # Chạy 1 seed chuẩn (đặt [42, 43, 44] khi chạy overnight commit)
EPOCHS = 15                  # 15 epochs là đủ cho mô hình hội tụ nhờ Early Stopping
PATIENCE = 3                 # Dừng sớm nếu sau 3 epochs Val MAE không giảm
BATCH_SIZE = 512             # Tận dụng tối đa bộ nhớ GPU P100/T4 (512 hoặc 1024)
NUM_WORKERS = 2 if torch.cuda.is_available() else 0  # Đọc dữ liệu song song trên GPU
SEQ_LEN, PRED_LEN = 12, 2
TARGET_TRANSFORM = 'log1p'
RETRAINED_ABLATION = False   # Đánh giá 5 kịch bản missing modality trên cùng 1 checkpoint chuẩn (Zero-shot)

print('📁 Data Dir:      ', DATA_DIR)
print('📁 Embeddings Dir:', EMBEDDINGS_DIR)


## 1. Chạy thực nghiệm Huấn luyện & Đánh giá Baselines
Thực hiện huấn luyện trên các mô hình baseline và đánh giá với 5 kịch bản missing modality:
- Full modalities (đủ ảnh + text + history)
- Mất ảnh (Missing Image)
- Mất mô tả (Missing Text)
- Chỉ còn nửa lịch sử gần nhất (Partial History)
- Mất cả ảnh và text (Missing Image + Text)

In [ ]:
from src.pipeline.train_baselines import run_all_baselines

RUN_DIR = run_all_baselines(
    epochs=EPOCHS,
    patience=PATIENCE,
    seeds=SEEDS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    processed_data_dir=DATA_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    output_dir=WORK_ROOT/'results/v2',
    models=MODELS,
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
    target_transform=TARGET_TRANSFORM,
    retrained_ablation=RETRAINED_ABLATION
)
print('✅ Run hoàn tất, kết quả lưu tại:', RUN_DIR)


## 2. Kết quả và Bảng So sánh Độ chính xác
Hiển thị bảng tổng hợp kết quả MAE, RMSE, wMAPE, sMAPE trên tập Test.

In [ ]:
import pandas as pd
from IPython.display import display

if (RUN_DIR/'baseline_results.csv').exists():
    print('=== KẾT QUẢ TỔNG QUAN CÁC BASELINES ===')
    display(pd.read_csv(RUN_DIR/'baseline_results.csv'))

if (RUN_DIR/'seed_summary.csv').exists():
    print('=== TỔNG HỢP THEO SEEDS (MEAN ± STD) ===')
    display(pd.read_csv(RUN_DIR/'seed_summary.csv'))

if (RUN_DIR/'robustness_comparison.csv').exists():
    print('=== BẢNG SO SÁNH CÁC KỊCH BẢN THIẾU DỮ LIỆU (MISSING MODALITIES) ===')
    display(pd.read_csv(RUN_DIR/'robustness_comparison.csv'))


## 3. Tải về Kết quả (Download Results)
Nén toàn bộ kết quả của Run thành file `.zip` để tải về máy dễ dàng.

In [ ]:
import shutil
from IPython.display import FileLink, display

archive_path = shutil.make_archive(str(WORK_ROOT/('baseline-results-'+RUN_DIR.name)), 'zip', root_dir=RUN_DIR)
print('✅ Đã tạo file nén kết quả tại:', archive_path)
display(FileLink(archive_path))
